# Fragrantica Dominant Accord Analysis

## Objective

This notebook analyzes the ordering of fragrance accords in the Fragrantica dataset to identify dominant accords and the most common combinations of the first two and first three accords.

Unlike an accord prevalence analysis, which examines all accords associated with each fragrance without considering their position, this analysis preserves the original ordering of accords. The first accord is treated as the **dominant accord**, followed by the secondary and tertiary accords, subject to validation of the source ordering.

## Analytical Scope

The analysis covers:

- **Dominant accord distribution:** Identify the most frequently occurring first accords across fragrances.
- **Top-two accord combinations:** Identify the most common ordered pairs of dominant and secondary accords.
- **Top-three accord combinations (triads):** Identify the most common ordered combinations of dominant, secondary, and tertiary accords.
- **Accord hierarchy patterns:** Explore how supporting accords are associated with dominant accords and how frequently specific ordered combinations occur.
- **Data preparation for recommendation systems:** Establish structured fragrance profiles that can support future fragrance wardrobe analysis, preference profiling, and personalized recommendations.

## Data Source

The analysis uses the cleaned Fragrantica reference dataset stored in Databricks:

`main.fragrance_analytics.fragrantica_perfumes`

The original source data will remain unchanged. Extracted accord positions, normalized values, and analytical results will be created as derived data.

## Methodology and Validation

Before analyzing accord dominance, the notebook inspects the source schema and sample records to establish how accords are stored and whether their original ordering is preserved.

The analysis then:

1. Validates the source representation and ordering of fragrance accords.
2. Extracts the first, second, and third accord positions without modifying the original data.
3. Checks for missing values, duplicate fragrance records, and inconsistent accord representations.
4. Calculates dominant accord frequencies and the distributions of ordered pairs and triads.
5. Examines the most frequent combinations and their supporting fragrance counts.
6. Documents relevant data limitations and sample sizes before interpreting the results.

The first accord is interpreted as dominant only if the source ordering supports this assumption. Ordered combinations remain directional: for example, *Woody → Amber* and *Amber → Woody* are treated as distinct pairs.

## Expected Outcomes

The notebook produces descriptive statistics and visualizations of dominant accord frequencies, the most common ordered pairs, and the most common triads. These outputs provide a structured view of fragrance accord hierarchies and a foundation for further rating analysis, gender comparisons in a separate notebook, and future fragrance recommendation functionality.

In [0]:

%pip install --upgrade "plotly>=6.1.1" "kaleido>=1.0.0"

from pyspark.sql import functions as F

# Load the cleaned Fragrantica reference dataset
source_table = "main.fragrance_analytics.fragrantica_perfumes"
df = spark.table(source_table)

# Exclude fragrances without accords for this analysis
df_with_accords = df.filter(
    F.col("main_accords").isNotNull()
    & (F.trim(F.col("main_accords")) != "[]")
)

# Check the source schema and sample records
df_with_accords.printSchema()
display(df_with_accords.limit(10))

# Validate the filtering results
display(
    df.agg(
        F.count("*").alias("original_fragrances")
    )
    .crossJoin(
        df_with_accords.agg(
            F.count("*").alias("fragrances_with_accords")
        )
    )
    .withColumn(
        "excluded_fragrances",
        F.col("original_fragrances") - F.col("fragrances_with_accords")
    )
)

In [0]:
from pyspark.sql import functions as F

# Parse the accord list into an array of strings
accord_array = F.from_json(
    F.regexp_replace(F.col("main_accords"), "'", '"'),
    "array<string>"
)

# Lowercase and trim each accord while preserving its position
standardized_accords = F.transform(
    accord_array,
    lambda accord: F.lower(F.trim(accord))
)

# Extract the dominant and secondary accords
dominant_accords_df = (
    df_with_accords
    .withColumn("accords", standardized_accords)
    .withColumn("accord_count", F.size("accords"))
    .withColumn("dominant_accord", F.get(F.col("accords"), 0))
    .withColumn("second_accord", F.get(F.col("accords"), 1))
    .withColumn(
        "top_2_profile",
        F.when(
            F.col("accord_count") >= 2,
            F.concat_ws(" → ", "dominant_accord", "second_accord")
        )
    )
)

# Inspect extracted accords and validate the standardization
display(
    dominant_accords_df.select(
        "name",
        "gender",
        "main_accords",
        "accords",
        "accord_count",
        "dominant_accord",
        "second_accord",
        "top_2_profile"
    ).limit(15)
)

In [0]:

from pyspark.sql import functions as F

# Extract distinct accords across all positions
all_accords = (
    dominant_accords_df
    .select(F.explode("accords").alias("accord"))
    .filter(F.col("accord").isNotNull() & (F.trim("accord") != ""))
    .distinct()
)

# Build distinct accord sets for positions 1–5
position_names = ["First", "Second", "Third", "Fourth", "Fifth"]

position_accords = {
    position: (
        dominant_accords_df
        .filter(F.size("accords") >= position)
        .select(F.col("accords")[position - 1].alias("accord"))
        .filter(F.col("accord").isNotNull() & (F.trim("accord") != ""))
        .distinct()
    )
    for position in range(1, 6)
}

# Count distinct accords in each position
summary_rows = [
    ("All distinct accords", all_accords.count())
]

for position in range(1, 6):
    summary_rows.append(
        (
            f"Distinct {position_names[position - 1].lower()} accords",
            position_accords[position].count()
        )
    )

# Count fragrances with at least two or three accords
two_accords_df = dominant_accords_df.filter(
    F.size("accords") >= 2
)

three_accords_df = dominant_accords_df.filter(
    F.size("accords") >= 3
)

fragrances_with_two_accords = two_accords_df.count()
fragrances_with_three_accords = three_accords_df.count()

# Count distinct ordered dominant–second pairs
distinct_pair_count = (
    two_accords_df
    .select("dominant_accord", "second_accord")
    .distinct()
    .count()
)

summary_rows.extend([
    ("Fragrances with at least two accords", fragrances_with_two_accords),
    ("Fragrances with at least three accords", fragrances_with_three_accords),
    ("Distinct dominant–second accord pairs", distinct_pair_count)
])

summary_df = spark.createDataFrame(
    summary_rows,
    ["metric", "unique_count"]
)

display(summary_df)

# Identify accords absent from each position compared with all accords
missing_accords_df = None

for position in range(1, 6):
    missing = (
        all_accords
        .join(position_accords[position], on="accord", how="left_anti")
        .withColumn("position", F.lit(position))
        .withColumn("position_name", F.lit(position_names[position - 1]))
    )

    missing_accords_df = (
        missing if missing_accords_df is None
        else missing_accords_df.unionByName(missing)
    )

display(
    missing_accords_df
    .select("position_name", "accord")
    .orderBy("position", "accord")
)

In [0]:

from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os

# ==========================================
# 1. Calculate the distribution
# ==========================================

valid_dominant_accords = dominant_accords_df.filter(
    F.col("dominant_accord").isNotNull()
)

total_valid_fragrances = valid_dominant_accords.count()

dominant_accord_distribution = (
    valid_dominant_accords
    .groupBy("dominant_accord")
    .agg(F.count("*").alias("fragrance_count"))
    .withColumn(
        "percentage",
        F.col("fragrance_count") / F.lit(total_valid_fragrances) * 100
    )
    .orderBy(F.desc("fragrance_count"))
)

display(
    dominant_accord_distribution
    .withColumn("percentage", F.round("percentage", 2))
    .limit(20)
)

# ==========================================
# 2. Prepare top 20 for plotting
# ==========================================

chart_df = (
    dominant_accord_distribution
    .limit(20)
    .toPandas()
    .sort_values("percentage", ascending=True)
    .reset_index(drop=True)
)

accords = chart_df["dominant_accord"]
percentages = chart_df["percentage"]
counts = chart_df["fragrance_count"]

# ==========================================
# 3. Plot: percentage-based x-axis
# ==========================================

background = "#21171f"
text_color = "#f4ece9"
muted_color = "#c6b6c0"
gold = "#d6b47a"
bar_color = "#98765b"
border_color = "#4d3948"

fig, ax = plt.subplots(figsize=(12, 9), dpi=180)
fig.patch.set_facecolor(background)
ax.set_facecolor(background)

colors = [
    gold if i == len(chart_df) - 1 else bar_color
    for i in range(len(chart_df))
]

bars = ax.barh(
    accords,
    percentages,
    color=colors,
    height=0.68,
    edgecolor="none",
    zorder=3
)

max_pct = float(percentages.max())
ax.set_xlim(0, max_pct * 1.30)

# Show percentage labels, with counts for additional context.
for bar, pct, count in zip(bars, percentages, counts):
    ax.text(
        bar.get_width() + max_pct * 0.018,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.2f}%  ({count:,.0f})",
        va="center",
        ha="left",
        color=text_color,
        fontsize=9
    )

fig.text(
    0.13, 0.965,
    "FRAGRANCE INTELLIGENCE  /  ACCORD ANALYSIS",
    color=gold,
    fontsize=9,
    fontweight="bold",
    ha="left"
)

fig.text(
    0.13, 0.925,
    "The dominant accord landscape",
    color=text_color,
    fontsize=23,
    fontfamily="serif",
    ha="left"
)

fig.text(
    0.13, 0.895,
    f"Top 20 accords  ·  {total_valid_fragrances:,} fragrances with a recorded dominant accord",
    color=muted_color,
    fontsize=10,
    ha="left"
)

ax.set_xlabel(
    "Share of fragrances (%)",
    color=muted_color,
    fontsize=10,
    labelpad=12,
    loc="center"
)
ax.set_ylabel("")

ax.xaxis.set_major_formatter(
    mtick.PercentFormatter(xmax=100, decimals=0)
)

ax.tick_params(axis="y", colors=text_color, labelsize=10, length=0)
ax.tick_params(axis="x", colors=muted_color, labelsize=9, length=0, pad=8)

ax.grid(
    axis="x",
    color=border_color,
    linewidth=0.8,
    alpha=0.7,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

# Center the bottom explanatory line
fig.text(
    0.5, 0.025,
    "Percentages use all fragrances with a non-null dominant accord. Bar labels show percentage and fragrance count.",
    color=muted_color,
    fontsize=8,
    ha="center"
)

plt.subplots_adjust(
    left=0.25,
    right=0.91,
    top=0.84,
    bottom=0.10
)

# ==========================================
# 4. Save the PNG locally in the notebook
# ==========================================

output_path = "/tmp/dominant-accord-distribution.png"

fig.savefig(
    output_path,
    dpi=220,
    facecolor=fig.get_facecolor(),
    bbox_inches="tight"
)

plt.show()

print(f"Chart saved to: {output_path}")

# Display the saved image in Databricks
displayHTML("""
<div style="margin: 16px 0;">
  <a href="files/dominant-accord-distribution.png"
     download="dominant-accord-distribution.png"
     style="color:#d6b47a;font-weight:600;">
     Download dominant accord distribution (PNG) ↗
  </a>
</div>
""")

In [0]:

# Count fragrances with at least two accords
two_accord_fragrance_count = (
    dominant_accords_df
    .filter(F.col("second_accord").isNotNull())
    .count()
)

# Calculate counts and percentages for all ordered accord pairs
accord_pair_distribution = (
    dominant_accords_df
    .filter(F.col("second_accord").isNotNull())
    .groupBy("dominant_accord", "second_accord")
    .agg(
        F.count("*").alias("fragrance_count")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("fragrance_count") / two_accord_fragrance_count * 100,
            2
        )
    )
    .orderBy(F.desc("fragrance_count"))
)

# Display all accord pairs with their counts and percentages
display(accord_pair_distribution)


In [0]:

# ==========================================
# Plot: accord pairs, matching previous chart
# ==========================================

import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

# Set this to the existing Databricks Volume directory
# where you store your other visualization PNGs.
OUTPUT_DIR = (
    "/Volumes/<catalog>/<schema>/<volume>/"
    "accord-analysis"
)

OUTPUT_FILENAME = "dominant-secondary-pairs.png"
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

chart_df = (
    accord_pair_distribution
    .orderBy(F.desc("fragrance_count"))
    .limit(20)
    .toPandas()
)

chart_df["accord_pair"] = (
    chart_df["dominant_accord"].str.title()
    + " → "
    + chart_df["second_accord"].str.title()
)

chart_df = (
    chart_df.sort_values("percentage", ascending=True)
    .reset_index(drop=True)
)

accord_pairs = chart_df["accord_pair"]
percentages = chart_df["percentage"]
counts = chart_df["fragrance_count"]

background = "#21171f"
text_color = "#f4ece9"
muted_color = "#c6b6c0"
gold = "#d6b47a"
bar_color = "#98765b"
border_color = "#4d3948"

# Exact same figure size and DPI as the previous visualization
fig, ax = plt.subplots(figsize=(12, 9), dpi=180)
fig.patch.set_facecolor(background)
ax.set_facecolor(background)

colors = [
    gold if i == len(chart_df) - 1 else bar_color
    for i in range(len(chart_df))
]

bars = ax.barh(
    accord_pairs,
    percentages,
    color=colors,
    height=0.68,
    edgecolor="none",
    zorder=3
)

max_pct = float(percentages.max())
ax.set_xlim(0, max_pct * 1.30)

for bar, pct, count in zip(bars, percentages, counts):
    ax.text(
        bar.get_width() + max_pct * 0.018,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.2f}%  ({count:,.0f})",
        va="center",
        ha="left",
        color=text_color,
        fontsize=9
    )

fig.text(
    0.13, 0.965,
    "FRAGRANCE INTELLIGENCE  /  ACCORD ANALYSIS",
    color=gold,
    fontsize=9,
    fontweight="bold",
    ha="left"
)

fig.text(
    0.13, 0.925,
    "The most common accord combinations",
    color=text_color,
    fontsize=23,
    fontfamily="serif",
    ha="left"
)

fig.text(
    0.13, 0.895,
    "Top 20 dominant–secondary accord pairs",
    color=muted_color,
    fontsize=10,
    ha="left"
)

ax.set_xlabel(
    "Share of fragrances (%)",
    color=muted_color,
    fontsize=10,
    labelpad=12,
    loc="center"
)
ax.set_ylabel("")

ax.xaxis.set_major_formatter(
    mtick.PercentFormatter(xmax=100, decimals=0)
)

ax.tick_params(axis="y", colors=text_color, labelsize=10, length=0)
ax.tick_params(axis="x", colors=muted_color, labelsize=9, length=0, pad=8)

ax.grid(
    axis="x",
    color=border_color,
    linewidth=0.8,
    alpha=0.7,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.text(
    0.5, 0.025,
    "Pairs are ordered dominant accord → secondary accord. Labels show percentage and fragrance count.",
    color=muted_color,
    fontsize=8,
    ha="center"
)

plt.subplots_adjust(
    left=0.29,
    right=0.91,
    top=0.84,
    bottom=0.10
)


# ==========================================
# Save chart to the Databricks Volume
# ==========================================

import os

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)

OUTPUT_FILENAME = "dominant-secondary-pairs.png"
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(
        f"Visualization directory does not exist: {OUTPUT_DIR}"
    )

fig.savefig(
    output_path,
    dpi=180,
    facecolor=fig.get_facecolor()
)

if not os.path.isfile(output_path):
    raise RuntimeError(f"Chart was not created: {output_path}")

print(f"Chart saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")
print(f"PNG dimensions: {fig.canvas.get_width_height()}")

plt.show()
plt.close(fig)

In [0]:

import plotly.express as px

# Prepare the top 20 complete accord pairs
plot_df = (
    accord_pair_distribution
    .filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
    )
    .orderBy(F.desc("fragrance_count"))
    .limit(20)
    .toPandas()
)

plot_df["dominant_label"] = plot_df["dominant_accord"].str.title()
plot_df["secondary_label"] = plot_df["second_accord"].str.title()
plot_df["pair_label"] = (
    plot_df["dominant_label"] + " → " + plot_df["secondary_label"]
)

# Create a treemap grouped by dominant and secondary accord
fig = px.treemap(
    plot_df,
    path=["dominant_label", "secondary_label"],
    values="fragrance_count",
    color="fragrance_count",
    color_continuous_scale=[
        "#F0E5E5", "#D4B5BF", "#B99AA5",
        "#9A6A78", "#674653", "#2F2930"
    ],
    custom_data=["pair_label", "percentage", "fragrance_count"]
)

fig.update_traces(
    texttemplate="%{label}<br>%{value:,}",
    marker=dict(line=dict(color="#F7F3EF", width=3)),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Fragrances: %{customdata[2]:,}<br>"
        "Share: %{customdata[1]:.2f}%"
        "<extra></extra>"
    ),
    tiling=dict(packing="squarify", pad=3)
)

fig.update_layout(
    title=dict(
        text="Dominant–Secondary Accord Profiles",
        x=0.03,
        font=dict(size=23, color="#2F2930")
    ),
    paper_bgcolor="#F7F3EF",
    font=dict(family="Arial", color="#2F2930"),
    margin=dict(t=85, l=15, r=15, b=20),
    height=720,
    coloraxis_showscale=False
)

displayHTML(
    fig.to_html(
        full_html=False,
        include_plotlyjs=True
    )
)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================
# 1. Parse and normalize accord labels
# ==========================================

# Parse accord labels while preserving their original order
accord_array = F.from_json(
    F.regexp_replace(F.col("main_accords"), "'", '"'),
    "array<string>"
)

accords_df = (
    df_with_accords
    .withColumn("accords", accord_array)
    .withColumn(
        "accords",
        F.transform(
            F.col("accords"),
            lambda x: F.lower(F.trim(x))
        )
    )
)

# ==========================================
# 2. Keep only valid three-level hierarchies
# ==========================================

accords_df = (
    accords_df
    .filter(
        F.col("accords").isNotNull()
        & (F.size("accords") >= 3)
        & F.col("accords")[0].isNotNull()
        & (F.trim(F.col("accords")[0]) != "")
        & F.col("accords")[1].isNotNull()
        & (F.trim(F.col("accords")[1]) != "")
        & F.col("accords")[2].isNotNull()
        & (F.trim(F.col("accords")[2]) != "")
    )
    .withColumn("dominant_accord", F.col("accords")[0])
    .withColumn("second_accord", F.col("accords")[1])
    .withColumn("third_accord", F.col("accords")[2])
)

# Validate the number of eligible fragrance records
eligible_fragrance_count = accords_df.count()

print(
    "Fragrances with three valid accord positions:",
    f"{eligible_fragrance_count:,}"
)

# ==========================================
# 3. Count three-level accord combinations
# ==========================================

accord_hierarchy = (
    accords_df
    .groupBy(
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
    .agg(F.count("*").alias("fragrance_count"))
)

# ==========================================
# 4. Calculate hierarchy totals and percentages
# ==========================================

accord_hierarchy = (
    accord_hierarchy
    .withColumn(
        "dominant_total",
        F.sum("fragrance_count").over(
            Window.partitionBy("dominant_accord")
        )
    )
    .withColumn(
        "pair_total",
        F.sum("fragrance_count").over(
            Window.partitionBy(
                "dominant_accord",
                "second_accord"
            )
        )
    )
    .withColumn(
        "second_pct_of_dominant",
        F.round(
            F.col("pair_total")
            / F.col("dominant_total") * 100,
            2
        )
    )
    .withColumn(
        "third_pct_of_pair",
        F.round(
            F.col("fragrance_count")
            / F.col("pair_total") * 100,
            2
        )
    )
)

# ==========================================
# 5. Validate the hierarchy before plotting
# ==========================================

display(
    accord_hierarchy
    .orderBy(
        F.desc("dominant_total"),
        F.desc("fragrance_count")
    )
    .limit(20)
)


In [0]:

import os
import re
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pyspark.sql import functions as F

# ==========================================
# 1. Retrieve and prepare hierarchy data
# ==========================================

def plot_accord_hierarchy(dominant_accord):

    pdf = (
        accord_hierarchy
        .filter(F.col("dominant_accord") == dominant_accord)
        .select(
            "second_accord",
            "third_accord",
            "fragrance_count"
        )
        .toPandas()
    )

    if pdf.empty:
        print(f"No data found for dominant accord: {dominant_accord}")
        return

    pdf["second_accord"] = (
        pdf["second_accord"].fillna("(no second accord)").astype(str)
    )
    pdf["third_accord"] = (
        pdf["third_accord"].fillna("(no third accord)").astype(str)
    )
    pdf["fragrance_count"] = pdf["fragrance_count"].astype(int)

    root_total = int(pdf["fragrance_count"].sum())

    if root_total == 0:
        print(f"No fragrances found for dominant accord: {dominant_accord}")
        return

    second_counts = (
        pdf.groupby("second_accord")["fragrance_count"]
        .sum()
        .sort_values(ascending=False)
    )

    second_top = second_counts.head(5)
    second_other = int(second_counts.iloc[5:].sum())

    second_items = []

    for second, pair_total in second_top.items():
        pair_total = int(pair_total)

        third_counts = (
            pdf.loc[pdf["second_accord"] == second]
            .groupby("third_accord")["fragrance_count"]
            .sum()
            .sort_values(ascending=False)
        )

        third_top = third_counts.head(5)
        third_other = int(third_counts.iloc[5:].sum())

        third_items = [
            {
                "name": str(name),
                "count": int(count),
                "pct": int(count) / pair_total * 100
            }
            for name, count in third_top.items()
        ]

        if third_other > 0:
            third_items.append({
                "name": "Other",
                "count": third_other,
                "pct": third_other / pair_total * 100
            })

        second_items.append({
            "name": str(second),
            "count": pair_total,
            "pct": pair_total / root_total * 100,
            "third": third_items
        })

    if second_other > 0:
        second_items.append({
            "name": "Other",
            "count": second_other,
            "pct": second_other / root_total * 100,
            "third": []
        })

    # ==========================================
    # 2. Visual style
    # ==========================================

    background = "#21171f"
    surface = "#30222d"
    text_color = "#f4ece9"
    muted_color = "#c6b6c0"
    gold = "#d6b47a"
    bar_color = "#98765b"
    border_color = "#4d3948"

    fig, ax = plt.subplots(figsize=(16, 12), dpi=180)

    fig.patch.set_facecolor(background)
    ax.set_facecolor(background)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # ==========================================
    # 3. Helper functions
    # ==========================================

    def add_card(
        x, y, width, height,
        facecolor=surface,
        edgecolor=border_color,
        radius=0.008,
        linewidth=1
    ):
        card = FancyBboxPatch(
            (x, y),
            width,
            height,
            boxstyle=f"round,pad=0.003,rounding_size={radius}",
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=linewidth,
            transform=ax.transAxes,
            zorder=2
        )
        ax.add_patch(card)

    def add_text(
        x, y, text,
        size=10,
        color=text_color,
        weight="normal",
        ha="center",
        va="center",
        family="sans-serif"
    ):
        ax.text(
            x, y, text,
            transform=ax.transAxes,
            fontsize=size,
            color=color,
            fontweight=weight,
            ha=ha,
            va=va,
            family=family,
            zorder=4
        )

    def add_line(
        x1, y1, x2, y2,
        color=border_color,
        linewidth=1.2
    ):
        ax.plot(
            [x1, x2],
            [y1, y2],
            transform=ax.transAxes,
            color=color,
            linewidth=linewidth,
            solid_capstyle="round",
            zorder=1
        )

    def shorten(text, max_chars=19):
        text = str(text)
        if len(text) > max_chars:
            return text[:max_chars - 1].rstrip() + "…"
        return text

    # ==========================================
    # 4. Title and subtitle
    # ==========================================

    fig.text(
        0.5, 0.965,
        "FRAGRANCE INTELLIGENCE  /  ACCORD ANALYSIS",
        color=gold,
        fontsize=9,
        fontweight="bold",
        ha="center"
    )

    fig.text(
        0.5, 0.932,
        f"{dominant_accord.title()} Accord Hierarchy",
        color=text_color,
        fontsize=23,
        fontfamily="serif",
        ha="center"
    )

    fig.text(
        0.5, 0.907,
        "Second-level share of dominant accord  ·  Third-level share of its parent",
        color=muted_color,
        fontsize=10,
        ha="center"
    )

    # ==========================================
    # 5. Root card
    # ==========================================

    root_x, root_y = 0.37, 0.815
    root_w, root_h = 0.26, 0.065

    add_card(
        root_x, root_y, root_w, root_h,
        facecolor=surface,
        edgecolor=gold,
        radius=0.012,
        linewidth=1.5
    )

    add_text(
        0.5, root_y + 0.044,
        shorten(dominant_accord.title(), 28),
        size=15,
        color=gold,
        weight="bold"
    )

    add_text(
        0.5, root_y + 0.019,
        f"{root_total:,} fragrances  ·  100%",
        size=10,
        color=text_color
    )

    # ==========================================
    # 6. Second-level branch layout
    # ==========================================

    n = len(second_items)

    centers = (
        [0.5] if n == 1
        else [0.09 + i * 0.82 / (n - 1) for i in range(n)]
    )

    col_width = min(0.145, 0.82 / max(n, 1))

    second_y = 0.695
    second_h = 0.085
    rail_y = 0.795

    # Root connector and horizontal rail
    add_line(0.5, root_y, 0.5, rail_y, color=gold, linewidth=1.4)

    if n > 1:
        add_line(
            min(centers), rail_y,
            max(centers), rail_y,
            color=gold,
            linewidth=1.2
        )

    # ==========================================
    # 7. Third-level layout: avoid card overlap
    # ==========================================

    child_h = 0.068
    child_gap = 0.012

    # Place the first child entirely below the second-level card.
    # This creates a clear vertical gap between both levels.
    child_top = second_y - 0.025 - child_h

    for i, item in enumerate(second_items):

        x = centers[i]
        x0 = x - col_width / 2

        # Connector to second-level branch
        add_line(
            x, rail_y,
            x, second_y + second_h,
            color=gold
        )

        # Second-level card
        add_card(
            x0, second_y, col_width, second_h,
            facecolor="#3b2b37",
            edgecolor=bar_color,
            radius=0.009
        )

        add_text(
            x, second_y + 0.062,
            shorten(item["name"], 20),
            size=10,
            color=text_color,
            weight="bold"
        )

        add_text(
            x, second_y + 0.039,
            f'{item["pct"]:.1f}% of root',
            size=9,
            color=gold
        )

        add_text(
            x, second_y + 0.017,
            f'n = {item["count"]:,}',
            size=9,
            color=muted_color
        )

        third = item["third"]

        if not third:
            add_text(
                x, second_y - 0.025,
                "Grouped remainder",
                size=8,
                color=muted_color
            )
            continue

        # Connector ends at the center of the first child card.
        connector_end = child_top + child_h / 2

        add_line(
            x, second_y,
            x, connector_end,
            color=border_color
        )

        for j, child in enumerate(third):

            # Stack child cards vertically with consistent spacing.
            cy = child_top - j * (child_h + child_gap)

            card_x = x0 + 0.003
            card_w = col_width - 0.006

            # Third-level card
            add_card(
                card_x, cy, card_w, child_h,
                facecolor=surface,
                edgecolor=border_color,
                radius=0.006,
                linewidth=0.8
            )

            # Separate text rows from the bottom indicator bar.
            add_text(
                x, cy + 0.052,
                shorten(child["name"], 19),
                size=9,
                color=text_color,
                weight="bold"
            )

            add_text(
                x, cy + 0.035,
                f'{child["pct"]:.1f}% of parent',
                size=8,
                color=gold
            )

            add_text(
                x, cy + 0.018,
                f'n = {child["count"]:,}',
                size=8,
                color=muted_color
            )

            # Proportional indicator bar, separated from the count.
            bar_x = card_x + 0.012
            bar_w = card_w - 0.024
            bar_y = cy + 0.005
            bar_h = 0.003

            add_card(
                bar_x, bar_y, bar_w, bar_h,
                facecolor=border_color,
                edgecolor=border_color,
                radius=0.001,
                linewidth=0
            )

            fraction = min(max(child["pct"] / 100, 0), 1)

            if fraction > 0:
                indicator_color = gold if j == 0 else bar_color

                add_card(
                    bar_x, bar_y, bar_w * fraction, bar_h,
                    facecolor=indicator_color,
                    edgecolor=indicator_color,
                    radius=0.001,
                    linewidth=0
                )

    # ==========================================
    # 8. Centered explanatory footer
    # ==========================================

    fig.text(
        0.5, 0.025,
        "Percentages are calculated within each parent group. Counts represent fragrances in each accord combination.",
        color=muted_color,
        fontsize=8,
        ha="center"
    )

    # ==========================================
    # 9. Save PNG
    # ==========================================

    plt.subplots_adjust(
        left=0.025,
        right=0.975,
        top=0.88,
        bottom=0.07
    )

    safe_name = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        dominant_accord.strip()
    ).strip("_").lower()

    output_path = f"/tmp/{safe_name}_accord_hierarchy.png"

    fig.savefig(
        output_path,
        format="png",
        dpi=220,
        facecolor=fig.get_facecolor()
    )

    plt.show()

    print(f"PNG saved to: {output_path}")
    print(f"PNG dimensions: {fig.canvas.get_width_height()}")

    # ==========================================
    # 10. Copy PNG to Unity Catalog Volume
    # ==========================================

    destination_dir = (
        "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
    )

    try:
        os.makedirs(destination_dir, exist_ok=True)

        destination = os.path.join(
            destination_dir,
            os.path.basename(output_path)
        )

        shutil.copy2(output_path, destination)

        print(f"PNG copied to Volume: {destination}")
        print(
            f"File size: "
            f"{os.path.getsize(destination) / (1024 * 1024):.2f} MB"
        )

    except Exception as e:
        print(f"PNG is saved locally, but Volume copy failed: {e}")

    return output_path


In [0]:
# plot_accord_hierarchy("woody")
# plot_accord_hierarchy("citrus")
# plot_accord_hierarchy("fruity")
# plot_accord_hierarchy("white floral")
# plot_accord_hierarchy("floral")
# plot_accord_hierarchy("aromatic")
# plot_accord_hierarchy("amber")
# plot_accord_hierarchy("warm spicy")
# plot_accord_hierarchy("sweet")
# plot_accord_hierarchy("vanilla")
# plot_accord_hierarchy("rose")
# plot_accord_hierarchy("powdery")
# plot_accord_hierarchy("musky")
# plot_accord_hierarchy("fresh spicy")
# plot_accord_hierarchy("green")
# plot_accord_hierarchy("oud")
# plot_accord_hierarchy("leather")
# plot_accord_hierarchy("lavender")
# plot_accord_hierarchy("patchouli")
# plot_accord_hierarchy("aquatic")

In [0]:

import os
import glob
import shutil

source_dir = "/tmp"

destination_dir = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)

os.makedirs(destination_dir, exist_ok=True)

# Find hierarchy HTML files and include the chart PNG.
html_files = glob.glob(
    os.path.join(source_dir, "*hierarchy*.html")
)

png_files = [
    os.path.join(source_dir, "dominant-accord-distribution.png")
]

# Combine files and keep only those that actually exist.
files_to_copy = sorted(
    set(
        path
        for path in html_files + png_files
        if os.path.isfile(path)
    )
)

if not files_to_copy:
    print("No matching HTML or PNG files found in /tmp.")

else:
    total_size = 0

    for source in files_to_copy:
        filename = os.path.basename(source)
        destination = os.path.join(destination_dir, filename)

        shutil.copy2(source, destination)

        size = os.path.getsize(destination)
        total_size += size

        print(
            f"Copied: {filename} "
            f"({size / (1024 * 1024):.2f} MB)"
        )

    print(f"\nFiles copied: {len(files_to_copy)}")
    print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
    print(f"Destination: {destination_dir}")

    print("\nFiles now available in the Volume:")
    for filename in sorted(os.listdir(destination_dir)):
        print(filename)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Dominant accords to analyze, ordered by overall frequency
accords_to_analyze = [
    "woody",
    "citrus",
    "fruity",
    "white floral",
    "floral",
    "aromatic",
    "amber",
    "warm spicy",
    "sweet",
    "vanilla",
    "rose",
    "powdery",
    "musky",
    "fresh spicy",
    "green",
    "oud",
    "leather",
    "lavender",
    "patchouli",
    "aquatic"
]

# --------------------------------------------------
# 1. DOMINANT ACCORD OVERVIEW
# --------------------------------------------------

dominant_overview = (
    accord_hierarchy
    .select("dominant_accord", "dominant_total")
    .distinct()
    .filter(F.col("dominant_accord").isin(accords_to_analyze))
    .withColumn(
        "share_of_three_accord_dataset_pct",
        F.round(
            F.col("dominant_total") / F.lit(eligible_fragrance_count) * 100,
            2
        )
    )
    .orderBy(F.desc("dominant_total"))
)

print("TABLE 1 — Dominant accord overview")
display(dominant_overview)


# --------------------------------------------------
# 2. MOST COMMON SECONDARY ACCORDS
#    AND THEIR MOST COMMON TERTIARY ACCORDS
# --------------------------------------------------

# One row per dominant-secondary pair
secondary_pairs = (
    accord_hierarchy
    .select(
        "dominant_accord",
        "second_accord",
        "dominant_total",
        "pair_total",
        "second_pct_of_dominant"
    )
    .distinct()
    .filter(F.col("dominant_accord").isin(accords_to_analyze))
)

# Rank secondary accords within each dominant accord
secondary_window = Window.partitionBy(
    "dominant_accord"
).orderBy(
    F.desc("pair_total"),
    F.asc("second_accord")
)

top_secondary = (
    secondary_pairs
    .withColumn(
        "secondary_rank",
        F.row_number().over(secondary_window)
    )
    .filter(F.col("secondary_rank") <= 5)
)

# Rank tertiary accords within each dominant-secondary pair
tertiary_window = Window.partitionBy(
    "dominant_accord", "second_accord"
).orderBy(
    F.desc("fragrance_count"),
    F.asc("third_accord")
)

top_tertiary = (
    accord_hierarchy
    .filter(F.col("dominant_accord").isin(accords_to_analyze))
    .join(
        top_secondary.select(
            "dominant_accord",
            "second_accord",
            "secondary_rank"
        ),
        on=["dominant_accord", "second_accord"],
        how="inner"
    )
    .withColumn(
        "tertiary_rank",
        F.row_number().over(tertiary_window)
    )
    .filter(F.col("tertiary_rank") <= 3)
)

# Combine into a single analysis table
accord_relationships = (
    top_tertiary
    .select(
        "dominant_accord",
        "dominant_total",
        "second_accord",
        "pair_total",
        "second_pct_of_dominant",
        "third_accord",
        "fragrance_count",
        "third_pct_of_pair",
        "secondary_rank",
        "tertiary_rank"
    )
    .withColumn(
        "third_pct_of_dominant",
        F.round(
            F.col("fragrance_count") / F.col("dominant_total") * 100,
            2
        )
    )
    .orderBy(
        F.expr(
            "array_position(array(" +
            ", ".join(f"'{a}'" for a in accords_to_analyze) +
            "), dominant_accord)"
        ),
        "secondary_rank",
        "tertiary_rank"
    )
)

print("TABLE 2 — Secondary and tertiary accord relationships")
display(accord_relationships)
